In [0]:
from datetime import date
from delta import DeltaTable
from pyspark.sql.functions import col ,current_timestamp,xxhash64,lit


In [0]:
%sql 
CREATE CATALOG IF NOT EXISTS nyctaxi;
USE CATALOG nyctaxi;
CREATE DATABASE if not EXISTS Inbound;
CREATE VOLUME IF NOT EXISTS nyctaxi.Inbound.inboundData;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS nyctaxi.inbound.nyc_taxi_inbound
(
    -- Identity / surrogate key
    row_id                  BIGINT     GENERATED ALWAYS AS IDENTITY,

    -- Source columns
    hvfhs_license_num       STRING,
    dispatching_base_num     STRING,
    originating_base_num    STRING,
    request_datetime        TIMESTAMP,
    on_scene_datetime       TIMESTAMP,
    pickup_datetime         TIMESTAMP,
    dropoff_datetime        TIMESTAMP,
    PULocationID            INT,
    DOLocationID            INT,
    trip_miles              DECIMAL(10, 2),
    trip_time               INT,
    base_passenger_fare     DECIMAL(10, 2),
    tolls                   DECIMAL(10, 2),
    bcf                     DECIMAL(10, 2),
    sales_tax               DECIMAL(10, 2),
    congestion_surcharge    DECIMAL(10, 2),
    airport_fee             DECIMAL(10, 2),
    tips                    DECIMAL(10, 2),
    driver_pay              DECIMAL(10, 2),
    shared_request_flag     STRING,
    shared_match_flag       STRING,
    access_a_ride_flag      STRING,
    wav_request_flag        STRING,
    wav_match_flag          STRING,
    cbd_congestion_fee      DECIMAL(10, 2),

    -- Generated column (partition key)
    pickup_date             DATE       GENERATED ALWAYS AS (CAST(pickup_datetime AS DATE)),

    -- Metadata
    created_on              TIMESTAMP  NOT NULL DEFAULT current_timestamp()
)
USING DELTA
PARTITIONED BY (pickup_date)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite'         = 'true',
    'delta.autoOptimize.autoCompact'            = 'true',
    'delta.feature.allowColumnDefaults'         = 'supported'
);